In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy import ndimage
import logging
import datetime

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

DISTANCE_THERSHOLD = 2.0 # mm
CONTACT_AREA_THRESHOLD = 10.0 # mm2
INTENSITY_DIFF_THRESHOLD = 0.2 # relative threshold
DILATION_RADIUS = 1 # voxels

logger.info(f"DISTANCE_THERSHOLD: {DISTANCE_THERSHOLD}")
logger.info(f"CONTACT_AREA_THRESHOLD: {CONTACT_AREA_THRESHOLD}")
logger.info(f"INTENSITY_DIFF_THRESHOLD: {INTENSITY_DIFF_THRESHOLD}")
logger.info(f"DILATION_RADIUS: {DILATION_RADIUS}")

In [ ]:
class DataLoader:
    def __init__(self, mri_path, annotation_path):
        self.mri_path = mri_path
        self.annotation_path = annotation_path

        self.mri_image = None
        self.annotation_image = None
        self.spacing = None
        self.num_slides = None
        self.node_labels = None
        self.node_masks = {}
        self.node_stats = {}
        self.adjacency_graph = None

        logger.info("DataLoader initialized")

    def load_data(self):
        """Load MRI and annotation data + some checking."""
        logger.info(f"Loading MRI image from {self.mri_path}")
        self.mri_image = sitk.ReadImage(self.mri_path)
        
        logger.info(f"Loading annotation image from {self.annotation_path}")
        self.annotation_image = sitk.ReadImage(self.annotation_path)

        # Ensure same coordinate system
        if not self._check_coordinate_match():
            logger.warning("MRI and annotation images might not be in the same coordinate system!")
        
        self.spacing = self.mri_image.GetSpacing()
        logger.info(f"Image spacing: {self.spacing}")
        
        # Extract node labels
        np_annotation = sitk.GetArrayFromImage(self.annotation_image)
        self.node_labels = np.unique(np_annotation)
        self.node_labels = self.node_labels[self.node_labels > 0]  # Remove background
        
        logger.info(f"Found {len(self.node_labels)} lymph node annotations with labels: {self.node_labels}")
        
        # Create individual masks for each node
        self._create_node_masks()
        
        return self
        
    def _check_coordinate_match(self):
        """Helper for the load_data function"""
        """Check if MRI and annotation images have matching coordinate systems."""
        mri_size = self.mri_image.GetSize()
        anno_size = self.annotation_image.GetSize()
        mri_spacing = self.mri_image.GetSpacing()
        anno_spacing = self.annotation_image.GetSpacing()
        mri_origin = self.mri_image.GetOrigin()
        anno_origin = self.annotation_image.GetOrigin()
        
        size_match = mri_size == anno_size
        spacing_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_spacing, anno_spacing))
        origin_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_origin, anno_origin))

        self.num_slides = anno_size[2]
        
        logger.info(f"Size match: {size_match}, Spacing match: {spacing_match}, Origin match: {origin_match}")
        logger.info(f"MRI spacing: {mri_spacing}, Anno spacing: {anno_spacing}")
        logger.info(f"xyz: {anno_size}, num_slides: {self.num_slides}")
        
        return size_match and spacing_match and origin_match
    
    def _create_node_masks(self):
        """Helper for the load_data function"""
        """Create binary masks for each lymph node."""
        for label in self.node_labels:
            logger.info(f"Creating mask for node {label}")
            
            # Create binary mask for this node
            node_mask = sitk.Equal(self.annotation_image, int(label))
            self.node_masks[label] = node_mask
            
            # Calculate basic statistics for this node
            np_mri = sitk.GetArrayFromImage(self.mri_image)
            np_mask = sitk.GetArrayFromImage(node_mask)
            node_voxels = np_mri[np_mask > 0]
            
            if len(node_voxels) > 0:
                self.node_stats[label] = {
                    'mean_intensity': np.mean(node_voxels),
                    'std_intensity': np.std(node_voxels),
                    'volume_mm3': np.sum(np_mask) * np.prod(self.spacing),
                    'voxel_count': np.sum(np_mask)
                }
                logger.info(f"  Node {label} stats: {self.node_stats[label]}")
            else:
                logger.warning(f"  Node {label} has no voxels!")


In [ ]:
class SliceAnalyzer:


In [ ]:
class MajorityCounter:

In [ ]:
class AnnotationMerger

In [ ]:
def run_pipeline_on_case(mri_path, annotation_path, output_path):
    dataloader = DataLoader(mri_path=mri_path, annotation_path=annotation_path)
    dataloader.load_data();

In [ ]:
if __name__ == "__main__":
    mri_path = "data/raw/images/1077-T2_FS_TRA+301.nii.gz"
    annotation_path = "data/raw/labels/1077-T2_FS_TRA+301.nii.gz"
    output_path = "test.nii.gz"

    run_pipeline_on_case(mri_path=mri_path, annotation_path=annotation_path, output_path=output_path)